<a href="https://colab.research.google.com/github/SyedaR06/CIS3120/blob/main/CIS3120_Rahman_Syeda_LibraryDB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
files.upload()
files.upload()
files.upload()


Saving Book.csv to Book.csv


Saving Loan.csv to Loan.csv


Saving Member.csv to Member.csv


{'Member.csv': b'id,firstname,lastName\r\n1,John,Smith\r\n2,David,Martin\r\n3,Betty,Freeman\r\n4,John,Martin\r\n'}

In [3]:
#step 1: setting up housekeeping, importing libraries and naming variables
import sqlite3
import csv
import os

book_path="Book.csv"
member_path="Member.csv"
loan_path="Loan.csv"

db_path="library.db"

In [4]:
# I skipped step two as there was no GitHub Repo for this project and the csv's were provided in the assignment on BrightSpace
#step 3: connect to the database
conn=sqlite3.connect(db_path)
conn.execute("PRAGMA foreign_keys=ON;")
print("Connected to the database!")

Connected to the database!


In [5]:
#step 3: create the tables
conn.execute("""
CREATE TABLE IF NOT EXISTS Book(
callNo TEXT NOT NULL,
title TEXT NOT NULL,
author TEXT NOT NULL,
PRIMARY KEY (callNo)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Member(
id INTEGER NOT NULL,
firstname TEXT NOT NULL,
lastName TEXT NOT NULL,
PRIMARY KEY (id)
);
""")

conn.execute("""
CREATE TABLE IF NOT EXISTS Loan(
callNo TEXT NOT NULL,
id INTEGER NOT NULL,
dateBorrowed TEXT NOT NULL,
dateReturned TEXT,
dateDue TEXT NOT NULL,
PRIMARY KEY (callNo, id, dateBorrowed),
FOREIGN KEY (callNo) REFERENCES Book(callNo),
FOREIGN KEY (id) REFERENCES Member(id)
);
""")

conn.commit()
print("Tables created.")

Tables created.


In [6]:
#Table verification
print(
    conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;").fetchall()
)

[('Book',), ('Loan',), ('Member',)]


In [7]:
#Step 4: Loading in Book.csv into the Book Table
with open(book_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Book (callNo, title, author) VALUES (?, ?, ?);",
            (row["callNo"], row["title"], row["author"])
        )

conn.commit()
print("Book rows loaded:", conn.execute("SELECT COUNT(*) FROM Book;").fetchone()[0])

Book rows loaded: 11


In [8]:
#Step 5: Loading in Member.csv into the Member Table
with open(member_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        conn.execute(
            "INSERT INTO Member (id, firstname, lastName) VALUES (?, ?, ?);",
            (int(row["id"]), row["firstname"], row["lastName"])
        )

conn.commit()
print("Member rows loaded:", conn.execute("SELECT COUNT(*) FROM Member;").fetchone()[0])

Member rows loaded: 4


In [9]:
#Step 6: Loading in Loan.csv into the Loan Table
with open(loan_path, newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        date_returned = row["dateReturned"] if row["dateReturned"].strip() else None

        conn.execute(
            """
            INSERT INTO Loan (callNo, id, dateBorrowed, dateReturned, dateDue)
            VALUES (?, ?, ?, ?, ?);
            """,
            (
                row["callNo"],
                int(row["id"]),
                row["dateBorrowed"],
                date_returned,
                row["dateDue"]
            )
        )

conn.commit()

print(
    "Loan rows loaded:",
    conn.execute("SELECT COUNT(*) FROM Loan;").fetchone()[0]
)

Loan rows loaded: 4


In [10]:
#query1-All Books
query1= """
SELECT *
FROM Book
ORDER BY author ASC;
"""

rows=conn.execute(query1).fetchall()
for r in rows:
    print(r)

('R 487 T35 1967', 'Medicine in medieval England.', 'Charles H Talbot')
('QA 76.9 D26H39 1996', 'Data model patterns : conventions of thought', 'David Hay')
('CB 351 M293 1983', 'Atlas of medieval Europe', 'Donald Matthew')
('HQ 1143 P68 1975', 'Medieval women', 'Eileen Power')
('PC 14 V48 1965', 'Medieval miscellany', 'Frederick Whitehead')
('QA 76.73 S67C435 2004', "Joe Celko's Trees and hierarchies in SQL for smarties", 'Joe Celko')
('QA 76.73 S67C46 1997', "Joe Celko's SQL puzzles & answers", 'Joe Celko')
('QA 76.9 D35C45 1999', "Joe Celko's data & databases : concepts in practice", 'Joe Celko')
('R 141 E45 2006', 'Medieval medicine and the plague', 'Lynne Elliott')
('QA 76.9 D26H355 2008', 'Information modeling and relational databases', 'T A Halpin')
('QA 76.76 A65P76 2011', 'Programming Android', 'Zigurd R Mednieks')


In [11]:
#query2-Books Not Yet Returned
query2= """
SELECT Book.title, Member.firstname, Member.lastName
FROM Loan
JOIN Book ON Loan.callNo=Book.callNo
JOIN Member ON Loan.id=Member.id
WHERE Loan.dateReturned is NULL;
"""

rows=conn.execute(query2).fetchall()
for r in rows:
    print(r)

("Joe Celko's SQL puzzles & answers", 'David', 'Martin')
('Medieval medicine and the plague', 'David', 'Martin')


In [12]:
#query3-Loan History For a Specific Book
query3= """
SELECT Member.firstname, Member.lastName, Loan.dateBorrowed, Loan.dateDue, Loan.dateReturned
FROM Loan
JOIN Member ON Loan.id=Member.id
WHERE Loan.callNo="R 141 E45 2006"
ORDER BY Loan.dateBorrowed ASC;
"""

rows=conn.execute(query3).fetchall()
for r in rows:
    print(r)

('Betty', 'Freeman', '4/1/2014 0:00', '4/15/2014 0:00', '4/15/2014 0:00')
('David', 'Martin', '4/30/2014 0:00', '5/14/2014 0:00', None)


In [13]:
#query4-Members Who Have Never Borrowed a Book
#this member is th
query4="""
SELECT Member.id, Member.firstname, Member.lastName
FROM Member
LEFT JOIN Loan ON Member.id =Loan.id
WHERE Loan.id IS NULL;
"""

rows=conn.execute(query4).fetchall()
for r in rows:
    print(r)

(4, 'John', 'Martin')


In [14]:
#query5
query5="""
SELECT Member.firstname, Member.lastName, Count(Loan.callNo) As total_loans
FROM Member
LEFT JOIN Loan ON Member.id =Loan.id
GROUP BY Member.id, Member.firstname, Member.lastName
ORDER BY total_loans DESC;
"""

rows=conn.execute(query5).fetchall()
for r in rows:
    print(r)

('David', 'Martin', 2)
('John', 'Smith', 1)
('Betty', 'Freeman', 1)
('John', 'Martin', 0)


In [15]:
#query6-What books have been repeatedly checked out?
query6="""
SELECT Book.title, Book.author, COUNT(DISTINCT Loan.id) AS repeatedly_checked_out_books
FROM Loan
JOIN Book ON Loan.callNo=Book.callNo
GROUP BY Book.callNo, Book.title, Book.author
HAVING COUNT(Loan.callNo) >1
ORDER BY repeatedly_checked_out_books DESC, Book.title ASC;
"""
rows=conn.execute(query6).fetchall()
for r in rows:
    print(r)

('Medieval medicine and the plague', 'Lynne Elliott', 2)


Markdown Summary:

One data quality observation I made through this Library Database Project is that there are some null values in the loan data set, specifically in the dataReturned column. This is because it represents books that have not yet been returned by the borrowers (members). When dealing with null cases, you have to be make sure that they are labeled as null and not the blanks/ empty strings they are saved in on the CSV's. One major limitation I noticed with this dataset is the lack of information for all three tables. While it makes sense for a small assignment, this does not accurately represent what an actual library system would look like, which would have thousands of rows of data for all categories. It would also definitely have additional columns for loan extensions, fines accumulated, member contact information, if books were lost/ damaged, etc.

Note: If the notebook doesn't work, delete the library.db in the folder, restart the kernel, then run all cells.

Note 2: I was unable to find the GitHub Repository, so instead, I just manually downloaded the CSV files provided on Brightspace and used them for this assignment.